# MAna NLP Module

`MAna.nlp` is the part of MAna that turns raw text into something a data
scientist can clean, split, vectorize, search, classify, and explain.

This notebook uses real text-heavy datasets:

- Women's clothing reviews for cleaning, sentiment, and supervised
  classification.
- TMDB movie overviews for chunking, topic modeling, and semantic search.
- YouTube trending-video metadata for short, noisy, title/tag-style text.

The goal is not to make the prettiest model. The goal is to show how each NLP
tool behaves in realistic situations: messy reviews, mixed numeric and text
features, short titles, long descriptions, and optional transformer workflows.

## 1. Setup

The notebook keeps everything local. Transformer and sentence-transformer
examples are included as optional cells because they may download model weights
the first time they run.

In [1]:
from pathlib import Path
import json

import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

from MAna.data import DataCleaner, read_data
from MAna.nlp import (
    TextCleaner,
    clean_text,
    clean_corpus,
    normalize_unicode,
    strip_accents,
    tokenize,
    DocumentChunk,
    TextChunker,
    chunk_text,
    chunk_documents,
    TextVectorizer,
    VectorizationResult,
    vectorize_texts,
    cosine_similarity_matrix,
    SentimentAnalyzer,
    SentimentResult,
    analyze_sentiment,
    TopicModeler,
    TopicModelResult,
    model_topics,
    TextEmbedder,
    EmbeddingResult,
    semantic_search,
    segment_text,
    encode_segmented_texts,
    build_text_classifier,
    evaluate_classifier,
)

pd.set_option("display.max_colwidth", 120)

DATA_DIR = Path("data")
if not DATA_DIR.exists():
    DATA_DIR = Path("docs/notebooks/data")
REVIEWS_PATH = DATA_DIR / "womens_clothing_reviews_sample.csv"
MOVIES_PATH = DATA_DIR / "tmdb_5000_movies.csv"
YOUTUBE_PATH = DATA_DIR / "youtube_trending_sample.csv"

## 2. Load Real Text Data

`read_data()` is used here instead of calling `pandas.read_csv()` directly.
That keeps the examples aligned with the data module and makes the notebook
easy to adapt when the same workflow moves from CSV to Excel, JSON, SQL, or
other supported sources.

In [2]:
# The three datasets represent three common NLP shapes:
# 1. customer-style reviews with labels and metadata,
# 2. longer narrative descriptions,
# 3. short, noisy social/video metadata.
reviews_raw = read_data(REVIEWS_PATH)
movies_raw = read_data(MOVIES_PATH)
youtube_raw = read_data(YOUTUBE_PATH)

{
    "reviews": reviews_raw.shape,
    "movies": movies_raw.shape,
    "youtube": youtube_raw.shape,
}

{'reviews': (160, 11), 'movies': (4803, 20), 'youtube': (20000, 15)}

In [3]:
# Keep only the columns needed for the notebook.
# This makes the examples readable while still preserving the real-world
# mixture of text, numeric, and categorical features.
reviews = (
    DataCleaner(reviews_raw, verbose=False)
    .standardize_column_names()
    .fix_missing_values(fill_value={"review_text": "", "title": ""})
    .fix_missing_values(
        strategy={
            "department_name": "mode",
            "class_name": "mode",
            "rating": "median",
            "age": "median",
        }
    )
    .coerce_numeric(columns=["rating", "recommended_ind", "age", "positive_feedback_count"])
    .drop_missing_rows(subset=["review_text"], treat_blank_as_missing=True)
    .get_cleaned_data()
)

reviews = reviews[
    [
        "review_text",
        "title",
        "rating",
        "recommended_ind",
        "age",
        "positive_feedback_count",
        "department_name",
        "class_name",
    ]
].copy()

# MAna.data has already enforced non-empty review text and explicit text defaults.
reviews["review_length"] = reviews["review_text"].str.split().str.len()

reviews.head()

,review_text,title,rating,recommended_ind,age,positive_feedback_count,department_name,class_name,review_length
0,Absolutely wonderful - silky and sexy and comfortable,,4,1,33,0,Intimate,Intimates,8
1,"Love this dress! it's sooo pretty. i happened to find it in a store, and i'm glad i did bc i never would have orde...",,5,1,34,4,Dresses,Dresses,62
2,I had such high hopes for this dress and really wanted it to work for me. i initially ordered the petite small (my u...,Some major design flaws,3,0,60,0,Dresses,Dresses,98
3,"I love, love, love this jumpsuit. it's fun, flirty, and fabulous! every time i wear it, i get nothing but great comp...",My favorite buy!,5,1,50,0,Bottoms,Pants,22
4,This shirt is very flattering to all due to the adjustable front tie. it is the perfect length to wear with leggings...,Flattering shirt,5,1,47,6,Tops,Blouses,36


In [4]:
# TMDB gives us longer documents. We keep rows with real overviews and parse
# the JSON-like genre strings into friendlier labels for interpretation.
movies = (
    DataCleaner(movies_raw, verbose=False)
    .coerce_numeric(columns=["vote_average", "vote_count"])
    .drop_missing_rows(subset=["overview"], treat_blank_as_missing=True)
    .get_cleaned_data()[["title", "overview", "genres", "vote_average", "vote_count"]]
)


def parse_genres(value): # a function to parse the genre JSON-like strings into a list of genre names
    try:
        parsed = json.loads(value)
    except (TypeError, json.JSONDecodeError):
        return []
    return [item.get("name", "") for item in parsed if item.get("name")]


movies["genre_names"] = movies["genres"].apply(parse_genres)
movies["primary_genre"] = movies["genre_names"].apply(
    lambda values: values[0] if values else "Unknown"
)
movies["overview_length"] = movies["overview"].str.split().str.len()

movies[["title", "primary_genre", "overview", "overview_length"]].head()

,title,primary_genre,overview,overview_length
0,Avatar,Action,"In the 22nd century, a paraplegic Marine is dispatched to the moon Pandora on a unique mission, but becomes torn bet...",28
1,Pirates of the Caribbean: At World's End,Adventure,"Captain Barbossa, long believed to be dead, has come back to life and is headed to the edge of the Earth with Will T...",34
2,Spectre,Action,A cryptic message from Bond’s past sends him on a trail to uncover a sinister organization. While M battles politica...,41
3,The Dark Knight Rises,Action,"Following the death of District Attorney Harvey Dent, Batman assumes responsibility for Dent's crimes to protect the...",65
4,John Carter,Action,"John Carter is a war-weary, former military captain who's inexplicably transported to the mysterious and exotic plan...",55


In [5]:
# YouTube titles and tags are short, noisy text. Standardizing column names
# makes the mixed casing from the source file easier to work with.
youtube = (
    DataCleaner(youtube_raw, verbose=False)
    .standardize_column_names()
    .fix_missing_values(fill_value={"title": "", "tags": ""})
    .fix_missing_values(
        strategy={
            "channel_title": "mode",
            "category_id": "mode",
            "views": "median",
            "likes": "median",
        }
    )
    .coerce_numeric(columns=["category_id", "views", "likes"], fill_value=0)
    .get_cleaned_data()
)

youtube["title_and_tags"] = youtube["title"] + " " + youtube["tags"].str.replace("|", " ")

youtube[["title", "channel_title", "category_id", "views", "likes", "title_and_tags"]].head()

,title,channel_title,category_id,views,likes,title_and_tags
0,Cheap Thrills - Sia / Tina Boo Choreography,1MILLION Dance Studio,24,601159,27962,"Cheap Thrills - Sia / Tina Boo Choreography choreography"" ""1million dance studio"" ""ìë°ë¦¬ì¸ ëì¤ ì¤íëì¤""..."
1,Cheap Thrills - Sia / Tina Boo Choreography,1MILLION Dance Studio,24,627933,28580,"Cheap Thrills - Sia / Tina Boo Choreography choreography"" ""1million dance studio"" ""ìë°ë¦¬ì¸ ëì¤ ì¤íëì¤""..."
2,FRIENDS - Marshmello & Anne-Marie / Tina Boo Choreography,1MILLION Dance Studio,24,384249,26271,"FRIENDS - Marshmello & Anne-Marie / Tina Boo Choreography choreography"" ""1million dance studio"" ""ìë°ë¦¬ì¸ ëì¤..."
3,FRIENDS - Marshmello & Anne-Marie / Tina Boo Choreography,1MILLION Dance Studio,24,513455,31505,"FRIENDS - Marshmello & Anne-Marie / Tina Boo Choreography choreography"" ""1million dance studio"" ""ìë°ë¦¬ì¸ ëì¤..."
4,FRIENDS - Marshmello & Anne-Marie / Tina Boo Choreography,1MILLION Dance Studio,24,607740,35180,"FRIENDS - Marshmello & Anne-Marie / Tina Boo Choreography choreography"" ""1million dance studio"" ""ìë°ë¦¬ì¸ ëì¤..."


## 3. Text Preprocessing

The preprocessing helpers are intentionally small and composable. Use the
single functions when you need one operation, and `TextCleaner` when you want a
repeatable cleaning policy for a whole corpus.

In [6]:
sample_review = reviews.loc[1, "review_text"]

# normalize_unicode() repairs common Unicode representation differences.
# strip_accents() is useful when matching text across systems that may or may
# not preserve diacritics.

{
    "original": sample_review,
    "normalized": normalize_unicode(sample_review),
    "without_accents": strip_accents("cafe naive facade"),
    "tokens": tokenize(sample_review)[:20],
}

{'original': 'Love this dress!  it\'s sooo pretty.  i happened to find it in a store, and i\'m glad i did bc i never would have ordered it online bc it\'s petite.  i bought a petite and am 5\'8".  i love the length on me- hits just a little below the knee.  would definitely be a true midi on someone who is truly petite.',
 'normalized': 'Love this dress!  it\'s sooo pretty.  i happened to find it in a store, and i\'m glad i did bc i never would have ordered it online bc it\'s petite.  i bought a petite and am 5\'8".  i love the length on me- hits just a little below the knee.  would definitely be a true midi on someone who is truly petite.',
 'without_accents': 'cafe naive facade',
 'tokens': ['Love',
  'this',
  'dress',
  "it's",
  'sooo',
  'pretty',
  'i',
  'happened',
  'to',
  'find',
  'it',
  'in',
  'a',
  'store',
  'and',
  "i'm",
  'glad',
  'i',
  'did',
  'bc']}

In [7]:
# clean_text() is the quick one-off function.
# Here we remove common English stop words but preserve negations by default,
# which matters for sentences such as "not good" or "never again".
clean_text(
    "I did NOT love the fit, but the fabric was very good!!! https://example.com",
    stop_words="english",
    preserve_negations=True,
)

'did not love fit fabric good'

In [8]:
# TextCleaner stores the policy, so the exact same rules can be reused across
# training, inference, topic modeling, and semantic search.
review_cleaner = TextCleaner(
    lowercase=True,
    remove_html=True,
    remove_urls=True,
    remove_emails=True,
    remove_punctuation=True,
    remove_numbers=False,
    strip_diacritics=True,
    stop_words="english",
    min_token_length=2,
    preserve_negations=True,
)

reviews["clean_review"] = review_cleaner.transform(reviews["review_text"])
reviews[["review_text", "clean_review"]].head(5)

,review_text,clean_review
0,Absolutely wonderful - silky and sexy and comfortable,absolutely wonderful silky sexy comfortable
1,"Love this dress! it's sooo pretty. i happened to find it in a store, and i'm glad i did bc i never would have orde...",love dress it's sooo pretty happened store i'm glad did bc never ordered online bc it's petite bought petite 5'8 lov...
2,I had such high hopes for this dress and really wanted it to work for me. i initially ordered the petite small (my u...,high hopes dress really wanted work initially ordered petite small usual size outrageously small small fact not zip ...
3,"I love, love, love this jumpsuit. it's fun, flirty, and fabulous! every time i wear it, i get nothing but great comp...",love love love jumpsuit it's fun flirty fabulous time wear great compliments
4,This shirt is very flattering to all due to the adjustable front tie. it is the perfect length to wear with leggings...,shirt flattering adjustable tie perfect length wear leggings sleeveless pairs cardigan love shirt


In [9]:
# clean_corpus() is the convenience form when you do not need to keep a cleaner
# instance around. It returns a list aligned to the input order.
cleaned_titles = clean_corpus(
    youtube["title"].head(10),
    stop_words="english",
    min_token_length=2,
    preserve_negations=True,
)

pd.DataFrame({"raw_title": youtube["title"].head(10), "clean_title": cleaned_titles})

,raw_title,clean_title
0,Cheap Thrills - Sia / Tina Boo Choreography,cheap thrills sia tina boo choreography
1,Cheap Thrills - Sia / Tina Boo Choreography,cheap thrills sia tina boo choreography
2,FRIENDS - Marshmello & Anne-Marie / Tina Boo Choreography,friends marshmello anne-marie tina boo choreography
3,FRIENDS - Marshmello & Anne-Marie / Tina Boo Choreography,friends marshmello anne-marie tina boo choreography
4,FRIENDS - Marshmello & Anne-Marie / Tina Boo Choreography,friends marshmello anne-marie tina boo choreography
5,FRIENDS - Marshmello & Anne-Marie / Tina Boo Choreography,friends marshmello anne-marie tina boo choreography
6,FRIENDS - Marshmello & Anne-Marie / Tina Boo Choreography,friends marshmello anne-marie tina boo choreography
7,FRIENDS - Marshmello & Anne-Marie / Tina Boo Choreography,friends marshmello anne-marie tina boo choreography
8,FRIENDS - Marshmello & Anne-Marie / Tina Boo Choreography,friends marshmello anne-marie tina boo choreography
9,This is me - The Greatest Showman OST / Jun Liu Choreography,greatest showman ost jun liu choreography


## 4. Chunking Documents

Chunking is needed before retrieval, summarization, long-document embedding, or
any workflow where a whole document is too large or too broad to represent as
one unit. MAna supports word, character, and sentence chunks, with overlap and
metadata preserved.

In [10]:
movie_row = movies.loc[movies["overview_length"].idxmax()]

# Word chunks are usually the safest first choice for natural-language search.
# Overlap preserves context at chunk boundaries.
word_chunks = chunk_text(
    movie_row["overview"],
    chunk_size=35,
    overlap=8,
    unit="words",
    metadata={"title": movie_row["title"], "primary_genre": movie_row["primary_genre"]},
)

pd.DataFrame([chunk.to_dict() for chunk in word_chunks]).head()

,text,index,start,end,document_index,metadata
0,"An orphaned girl, driven by poverty at such a young age, makes a promise with an enchantress. In return for beauty a...",0,0,182,0,"{'title': 'The Promise', 'primary_genre': 'Fantasy'}"
1,she will never be with the man she loves. This spell cannot be broken unless the impossible happens: snow falling in...,1,148,341,0,"{'title': 'The Promise', 'primary_genre': 'Fantasy'}"
2,"to life. Now a grown and beautiful princess, she regrets her promise, for all of the men she's loved has always been...",2,297,475,0,"{'title': 'The Promise', 'primary_genre': 'Fantasy'}"
3,"again with a man behind a red armor and a golden mask who rescues her from death, she is tormented by their inevitab...",3,440,633,0,"{'title': 'The Promise', 'primary_genre': 'Fantasy'}"
4,"slave of a great general, is searching for the lost memories of a family he once had. Soon the fate of these two int...",4,591,774,0,"{'title': 'The Promise', 'primary_genre': 'Fantasy'}"


In [11]:
# Character chunks are useful for fixed-width systems or legacy APIs.
# Sentence chunks are useful when sentence boundaries matter more than token
# counts. Both return the same DocumentChunk structure.
character_chunks = chunk_text(
    movie_row["overview"],
    chunk_size=220,
    overlap=40,
    unit="characters",
    metadata={"title": movie_row["title"]},
)

sentence_chunks = chunk_text(
    movie_row["overview"],
    chunk_size=2,
    overlap=1,
    unit="sentences",
    metadata={"title": movie_row["title"]},
)

{
    "character_chunks": len(character_chunks),
    "sentence_chunks": len(sentence_chunks),
    "first_sentence_chunk": sentence_chunks[0].to_dict() if sentence_chunks else None,
}

{'character_chunks': 6,
 'sentence_chunks': 8,
 'first_sentence_chunk': {'text': 'An orphaned girl, driven by poverty at such a young age, makes a promise with an enchantress. In return for beauty and the admiration of every man, she will never be with the man she loves.',
  'index': 0,
  'start': 0,
  'end': 189,
  'document_index': 0,
  'metadata': {'title': 'The Promise'}}}

In [12]:
# TextChunker is the reusable transformer-style wrapper around chunk_documents().
# Metadata stays attached, which is important when chunks later become search
# results and need to point back to their source document.
chunker = TextChunker(chunk_size=45, overlap=10, unit="words") # to create a TextChunker instance that will split text into chunks of 45 words with an overlap of 10 words between chunks. This is useful for processing longer documents while preserving context across chunk boundaries.
movie_subset = movies.head(5).copy()

chunked_movies = chunker.transform(
    movie_subset["overview"].tolist(),
    metadata=movie_subset[["title", "primary_genre"]].to_dict("records"),
)

pd.DataFrame([chunk.to_dict() for chunk in chunked_movies]).head(10)

,text,index,start,end,document_index,metadata
0,"In the 22nd century, a paraplegic Marine is dispatched to the moon Pandora on a unique mission, but becomes torn bet...",0,0,175,0,"{'title': 'Avatar', 'primary_genre': 'Action'}"
1,"Captain Barbossa, long believed to be dead, has come back to life and is headed to the edge of the Earth with Will T...",0,0,176,1,"{'title': 'Pirates of the Caribbean: At World's End', 'primary_genre': 'Adventure'}"
2,A cryptic message from Bond’s past sends him on a trail to uncover a sinister organization. While M battles politica...,0,0,240,2,"{'title': 'Spectre', 'primary_genre': 'Action'}"
3,"Following the death of District Attorney Harvey Dent, Batman assumes responsibility for Dent's crimes to protect the...",0,0,303,3,"{'title': 'The Dark Knight Rises', 'primary_genre': 'Action'}"
4,"the mysterious Selina Kyle and the villainous Bane, a new terrorist leader who overwhelms Gotham's finest. The Dark ...",1,246,428,3,"{'title': 'The Dark Knight Rises', 'primary_genre': 'Action'}"
5,"John Carter is a war-weary, former military captain who's inexplicably transported to the mysterious and exotic plan...",0,0,287,4,"{'title': 'John Carter', 'primary_genre': 'Action'}"
6,"collapse, and Carter rediscovers his humanity when he realizes the survival of Barsoom and its people rests in his h...",1,221,342,4,"{'title': 'John Carter', 'primary_genre': 'Action'}"


## 5. Vectorization

Vectorization converts text into numerical features. MAna exposes TF-IDF,
count vectors, and hashing vectors through one interface, while preserving
feature names and helper methods for inspection.

In [13]:
# TF-IDF is a strong default for medium-sized text tables because it downweights
# words that appear everywhere and highlights words that distinguish documents.
review_vectorizer = TextVectorizer(
    method="tfidf",
    cleaner=review_cleaner,
    max_features=1200,
    ngram_range=(1, 2),
    min_df=1,
)

review_vectors = review_vectorizer.fit_transform(reviews["review_text"])

{
    "matrix_shape": review_vectors.shape,
    "top_terms_first_review": review_vectors.top_terms(0, n=8),
}

{'matrix_shape': (139, 1200),
 'top_terms_first_review': [('wonderful', 0.5725691271570721),
  ('sexy', 0.5725691271570721),
  ('absolutely', 0.49062153129488784),
  ('comfortable', 0.3219001433411242)]}

In [14]:
# to_dataframe(dense=True) is fine for a small inspection slice.
# Keep dense=False for real modeling because sparse matrices are much more
# memory-friendly for text data.
review_vectors.to_dataframe(dense=True).iloc[:5, :10]

,10,120,125,135,30,36,36d,38,absolutely,actually
0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.490622,0.0
1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0
2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0
3,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0
4,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0


In [15]:
# transform() applies the trained vocabulary to new text. This is the right
# pattern for production inference because the vocabulary is not refit.
new_review_vectors = review_vectorizer.transform(
    [
        "The fabric is beautiful and the fit is comfortable.",
        "The zipper broke and the sizing was not good.",
    ]
)

new_review_vectors.shape

(2, 1200)

In [16]:
# Count vectors are easier to explain when you need raw term frequency.
# This example uses YouTube titles and tags, where repeated words and phrases
# can reveal content themes.
youtube_count_vectors = vectorize_texts(
    youtube["title_and_tags"].head(1000),
    method="count",
    max_features=500,
    ngram_range=(1, 2),
    stop_words="english",
)

{
    "matrix_shape": youtube_count_vectors.shape,
    "top_terms_first_video": youtube_count_vectors.top_terms(0, n=10),
}

{'matrix_shape': (1000, 500),
 'top_terms_first_video': [('dance', 3.0),
  ('choreography', 2.0),
  ('1million', 2.0),
  ('urban', 1.0),
  ('studio', 1.0),
  ('music urban', 1.0),
  ('music', 1.0),
  ('hiphop music', 1.0),
  ('1million dance', 1.0)]}

In [17]:
# Hashing vectors do not keep a learned vocabulary, so they are useful when
# data arrives as a stream or the vocabulary is too large to store directly.
hashed_titles = vectorize_texts(
    youtube["title"].head(500),
    method="hashing",
    n_features=128,
    alternate_sign=False,
)

hashed_titles.shape

(500, 128)

In [18]:
# cosine_similarity_matrix() works with vectorizer output or any compatible
# sparse/dense matrix. Here it tells us which review snippets are lexically
# close to each other after cleaning.
similarity = cosine_similarity_matrix(review_vectors.matrix[:8])

pd.DataFrame(
    similarity,
    index=[f"review_{i}" for i in range(8)],
    columns=[f"review_{i}" for i in range(8)],
).round(2)

,review_0,review_1,review_2,review_3,review_4,review_5,review_6,review_7
review_0,1.00,0.00,0.03,0.00,0.00,0.00,0.00,0.00
review_1,0.00,1.00,0.08,0.10,0.05,0.13,0.05,0.05
review_2,0.03,0.08,1.00,0.00,0.00,0.10,0.03,0.04
review_3,0.00,0.10,0.00,1.00,0.08,0.08,0.00,0.00
review_4,0.00,0.05,0.00,0.08,1.00,0.03,0.00,0.00
review_5,0.00,0.13,0.10,0.08,0.03,1.00,0.02,0.04
review_6,0.00,0.05,0.03,0.00,0.00,0.02,1.00,0.35
review_7,0.00,0.05,0.04,0.00,0.00,0.04,0.35,1.00


## 6. Sentiment Analysis

`SentimentAnalyzer` can run with a lightweight MAna lexicon, VADER, or a
transformer pipeline. The lexicon backend is intentionally dependency-light,
so it is good for fast exploration and documentation examples.

In [19]:
sentiment = SentimentAnalyzer(
    backend="lexicon",
    positive_threshold=0.05,
    negative_threshold=-0.05,
)

sentiment_examples = sentiment.to_dataframe(reviews["review_text"].head(20))
sentiment_examples[["label", "score", "positive", "neutral", "negative", "text"]].head(10)

,label,score,positive,neutral,negative,text
0,positive,0.600000,0.600000,0.400000,0.0,Absolutely wonderful - silky and sexy and comfortable
1,positive,0.707107,0.707107,0.292893,0.0,"Love this dress! it's sooo pretty. i happened to find it in a store, and i'm glad i did bc i never would have orde..."
2,neutral,0.000000,0.000000,1.000000,-0.0,I had such high hopes for this dress and really wanted it to work for me. i initially ordered the petite small (my u...
3,positive,0.894427,0.894427,0.105573,0.0,"I love, love, love this jumpsuit. it's fun, flirty, and fabulous! every time i wear it, i get nothing but great comp..."
4,positive,0.707107,0.707107,0.292893,0.0,This shirt is very flattering to all due to the adjustable front tie. it is the perfect length to wear with leggings...
5,positive,0.707107,0.707107,0.292893,0.0,"I love tracy reese dresses, but this one is not for the very petite. i am just under 5 feet tall and usually wear a ..."
6,neutral,0.000000,0.000000,1.000000,-0.0,I aded this in my basket at hte last mintue to see what it would look like in person. (store pick up). i went with t...
7,neutral,0.000000,0.000000,1.000000,-0.0,"I ordered this in carbon for store pick up, and had a ton of stuff (as always) to try on and used this top to pair (..."
8,positive,0.447214,0.447214,0.552786,0.0,I love this dress. i usually get an xs but it runs a little snug in bust so i ordered up a size. very flattering and...
9,neutral,0.000000,0.000000,1.000000,-0.0,"I'm 5""5' and 125 lbs. i ordered the s petite to make sure the length wasn't too long. i typically wear an xs regular..."


In [20]:
# analyze_sentiment() is the convenience wrapper. It accepts one text value or
# a sequence of values.
single_result = analyze_sentiment(
    "The dress is not perfect, but the material is good and comfortable."
)

batch_results = analyze_sentiment(
    [
        "I absolutely love the quality.",
        "The zipper failed and the fabric feels weak.",
        "The color is fine.",
    ]
)

{
    "single": single_result.to_dict(),
    "batch": [result.to_dict() for result in batch_results],
}

{'single': {'text': 'The dress is not perfect, but the material is good and comfortable.',
  'label': 'neutral',
  'score': 0.0,
  'positive': 0.0,
  'neutral': 1.0,
  'negative': -0.0,
  'backend': 'lexicon'},
 'batch': [{'text': 'I absolutely love the quality.',
   'label': 'positive',
   'score': 0.6,
   'positive': 0.6,
   'neutral': 0.4,
   'negative': 0.0,
   'backend': 'lexicon'},
  {'text': 'The zipper failed and the fabric feels weak.',
   'label': 'negative',
   'score': -0.7071067811865475,
   'positive': 0.0,
   'neutral': 0.29289321881345254,
   'negative': 0.7071067811865475,
   'backend': 'lexicon'},
  {'text': 'The color is fine.',
   'label': 'neutral',
   'score': 0.0,
   'positive': 0.0,
   'neutral': 1.0,
   'negative': -0.0,
   'backend': 'lexicon'}]}

In [21]:
# Joining sentiment back to the source table creates a quick diagnostic view:
# Where does the text sentiment agree or disagree with the user's rating?
review_sentiment = sentiment.to_dataframe(reviews["review_text"])
sentiment_audit = reviews.join(
    review_sentiment[["label", "score"]].rename(
        columns={"label": "sentiment_label", "score": "sentiment_score"}
    )
)

sentiment_audit.groupby(["recommended_ind", "sentiment_label"]).size().unstack(fill_value=0)

sentiment_label,negative,neutral,positive
recommended_ind,,,
0,1,12,11
1,7,34,54


In [22]:
# Optional VADER backend. This cell is safe on machines without vaderSentiment:
# it reports the install note instead of breaking the notebook.
try:
    vader_result = SentimentAnalyzer(backend="vader").analyze(reviews.loc[0, "review_text"])
    vader_result.to_dict()
except ImportError as exc:
    str(exc)

## 7. Topic Modeling

Topic modeling helps summarize a corpus without labels. NMF on TF-IDF tends to
work well for interpretable themes, while LDA on counts is a useful classical
alternative.

In [23]:
# Movie overviews are a better topic-modeling dataset than short reviews
# because each document contains enough story detail for themes to separate.
movie_docs = (
    DataCleaner(movies, verbose=False)
    .drop_missing_rows(subset=["overview"], treat_blank_as_missing=True)
    .get_cleaned_data()
    .head(900)
)

movie_topic_model = TopicModeler(
    n_topics=6,
    method="nmf",
    n_top_words=9,
    max_features=1800,
    cleaner=TextCleaner(stop_words="english", min_token_length=3),
    vectorizer_kwargs={"min_df": 3, "ngram_range": (1, 2)},
    random_state=42,
)

movie_topics = movie_topic_model.fit_transform(movie_docs["overview"])
movie_topics.topics_frame()

TypeError: unhashable type: 'list'

In [ ]:
# assignments() gives each document's dominant topic and confidence.
# Joining titles back makes the topic labels easier to judge.
movie_assignments = movie_topics.assignments()
movie_assignments = movie_assignments.join(movie_docs[["title", "primary_genre"]].reset_index(drop=True))

movie_assignments[["title", "primary_genre", "topic", "topic_label", "confidence"]].head(15)

,title,primary_genre,topic,topic_label,confidence
0,Avatar,Action,4,earth / planet / race,0.093153
1,Pirates of the Caribbean: At World's End,Adventure,4,earth / planet / race,0.122179
2,Spectre,Action,2,agent / team / fbi,0.120846
3,The Dark Knight Rises,Action,3,new / york / new york,0.140116
4,John Carter,Action,1,world / war / world war,0.125876
5,Spider-Man 3,Fantasy,0,life / man / old,0.073175
6,Tangled,Animation,0,life / man / old,0.043478
7,Avengers: Age of Ultron,Action,4,earth / planet / race,0.151473
8,Harry Potter and the Half-Blood Prince,Adventure,0,life / man / old,0.081931
9,Batman v Superman: Dawn of Justice,Action,1,world / war / world war,0.104227


In [ ]:
# transform() applies the fitted topic model to new documents.
# The output columns correspond to the learned topic ids.
new_movie_topic_weights = movie_topic_model.transform(
    [
        "A detective follows a dangerous criminal conspiracy across the city.",
        "A crew travels through space and discovers an alien civilization.",
    ]
)

pd.DataFrame(
    new_movie_topic_weights,
    columns=[f"topic_{topic.id}: {topic.label}" for topic in movie_topics.topics],
).round(3)

,topic_0: life / man / old,topic_1: world / war / world war,topic_2: agent / team / fbi,topic_3: new / york / new york,topic_4: earth / planet / race,topic_5: story / true / true story
0,0.0,0.003,0.036,0.084,0.004,0.012
1,0.0,0.000,0.000,0.000,0.175,0.010


In [ ]:
# LDA uses count vectors instead of TF-IDF. On YouTube title/tag text, it gives
# a different view of short-form content themes.
youtube_topic_docs = youtube["title_and_tags"].head(1500)

youtube_lda_topics = model_topics(
    youtube_topic_docs,
    n_topics=5,
    method="lda",
    n_top_words=8,
    max_features=1000,
    cleaner=TextCleaner(stop_words="english", min_token_length=2),
    vectorizer_kwargs={"min_df": 5, "ngram_range": (1, 2)},
    model_kwargs={"max_iter": 10},
)

youtube_lda_topics.topics_frame()

,id,label,terms,weights
0,0,2018 / bbc / walker,"[2018, bbc, walker, match, alan, alan walker, dance, planet]","[178.55223622160054, 153.27019721650208, 99.19999733770467, 83.19997595882923, 80.19999735879296, 80.19999735879294,..."
1,1,news / bbc / amazon,"[news, bbc, amazon, trailer, black, video, black panther, panther]","[439.76249304719437, 229.61158506617224, 154.2113030254598, 122.41380061476445, 121.13545268345176, 109.486705209215..."
2,2,music / official / a24,"[music, official, a24, trailer, movie, bright, album, andrew]","[201.4132961638754, 137.01739031733337, 128.19965467065077, 114.2677571013788, 92.5570490793774, 81.19971541831019, ..."
3,3,makeup / bowl / 2017,"[makeup, bowl, 2017, tutorial, makeup tutorial, 2018, nick, super]","[212.1986244109517, 110.06776846123502, 104.5965424089499, 104.0381757359552, 95.19875192222413, 90.94662339451374, ..."
4,4,anthony / padilla / anthony padilla,"[anthony, padilla, anthony padilla, iphone, smosh, youtube, padilla anthony, apple]","[320.19939246592475, 269.199370225139, 225.1993357536473, 181.19915450138512, 114.19912930017205, 110.65715721452945..."


## 8. Embeddings and Semantic Search

MAna supports a lightweight TF-IDF embedder and optional
sentence-transformer embeddings. The TF-IDF backend is useful for local,
repeatable examples. Sentence-transformers are stronger semantically when the
extra dependency and model weights are available.

In [ ]:
# TextEmbedder gives a fit/encode interface around embeddings.
# With backend='tfidf', the embedding space is learned from this movie corpus.
movie_embedder = TextEmbedder(
    backend="tfidf",
    cleaner=TextCleaner(stop_words="english", min_token_length=2),
    max_features=2200,
    ngram_range=(1, 2),
)

movie_embeddings = movie_embedder.fit_transform(movie_docs["overview"].head(600))

{
    "embedding_shape": movie_embeddings.shape,
    "backend": movie_embeddings.backend,
    "model_name": movie_embeddings.model_name,
}

{'embedding_shape': (600, 2200), 'backend': 'tfidf', 'model_name': 'tfidf'}

In [ ]:
# search() ranks an already embedded corpus against a query vector.
query_vector = movie_embedder.encode(["alien planet space marine civilization"]).vectors

movie_embeddings.search(query_vector, top_k=5)

,index,text,score
0,0,22nd century paraplegic marine dispatched moon pandora unique mission torn following orders protecting alien civiliz...,0.424961
1,461,prospects continuing life earth year 2058 grim robinsons launched space colonize alpha prime inhabitable planet gala...,0.258362
2,581,alien race factions starfleet attempt planet regenerative properties falls captain picard crew enterprise defend pla...,0.256643
3,27,mankind beams radio signal space reply comes planet form alien crafts splash waters hawaii lieutenant alex hopper we...,0.240707
4,507,july giant alien mothership enters orbit earth deploys dozen saucer-shaped destroyer spacecraft quickly lay waste ma...,0.229775


In [ ]:
# semantic_search() is the one-call version. It fits the default TF-IDF
# embedder when one is not supplied.
search_results = semantic_search(
    "undercover agent criminal conspiracy city",
    movie_docs["overview"].head(600).tolist(),
    top_k=5,
)

search_results

,index,text,score
0,308,A young undercover FBI agent infiltrates a gang of thieves who share a common interest in extreme sports. A remake o...,0.179963
1,442,An antiterrorism agent goes under the knife to acquire the likeness of a terrorist and gather details about a bombin...,0.142629
2,351,"To take down South Boston's Irish Mafia, the police send in one of their own to infiltrate the underworld, not reali...",0.142246
3,422,Futuristic action about a man who meets a clone of himself and stumbles into a grand conspiracy about clones taking ...,0.127279
4,531,"At the height of the Cold War, a mysterious criminal organization plans to use nuclear weapons and technology to ups...",0.124444


In [ ]:
# Add movie titles back to the semantic-search results for a readable report.
search_results.assign(
    title=search_results["index"].map(movie_docs.head(600).reset_index(drop=True)["title"]),
    primary_genre=search_results["index"].map(movie_docs.head(600).reset_index(drop=True)["primary_genre"]),
)[["title", "primary_genre", "score", "text"]]

,title,primary_genre,score,text
0,Point Break,Action,0.179963,A young undercover FBI agent infiltrates a gang of thieves who share a common interest in extreme sports. A remake o...
1,Face/Off,Action,0.142629,An antiterrorism agent goes under the knife to acquire the likeness of a terrorist and gather details about a bombin...
2,The Departed,Drama,0.142246,"To take down South Boston's Irish Mafia, the police send in one of their own to infiltrate the underworld, not reali..."
3,The 6th Day,Science Fiction,0.127279,Futuristic action about a man who meets a clone of himself and stumbles into a grand conspiracy about clones taking ...
4,The Man from U.N.C.L.E.,Comedy,0.124444,"At the height of the Cold War, a mysterious criminal organization plans to use nuclear weapons and technology to ups..."


In [ ]:
# segment_text() is a simple way to split one document into a fixed number of
# contiguous sections. It is helpful when the beginning, middle, and end of a
# document should keep their position.
segment_text(movie_docs.iloc[0]["overview"], n_segments=4)

['In the 22nd century, a paraplegic Marine',
 'is dispatched to the moon Pandora on',
 'a unique mission, but becomes torn between',
 'following orders and protecting an alien civilization.']

In [ ]:
# encode_segmented_texts() normally uses sentence-transformers. To keep this
# notebook runnable without downloading model weights, this demonstration uses
# a tiny local encoder adapter on real movie text. The important behavior shown
# here is MAna's fixed-section aggregation: concatenate preserves section
# position, while mean collapses sections into one vector width.
class LocalLengthEncoder:
    def encode(self, texts, **kwargs):
        return np.asarray(
            [
                [len(text), len(text.split()), text.lower().count("love")]
                for text in texts
            ],
            dtype=float,
        )


segmented_concat = encode_segmented_texts(
    movie_docs["overview"].head(6).tolist(),
    n_segments=3,
    encoder=LocalLengthEncoder(),
    aggregation="concatenate",
)

segmented_mean = encode_segmented_texts(
    movie_docs["overview"].head(6).tolist(),
    n_segments=3,
    encoder=LocalLengthEncoder(),
    aggregation="mean",
)

{
    "concatenate_shape": segmented_concat.shape,
    "mean_shape": segmented_mean.shape,
}

{'concatenate_shape': (6, 9), 'mean_shape': (6, 3)}

## 9. Supervised Text Classification

`build_text_classifier()` creates a leakage-safe scikit-learn pipeline. The
TF-IDF vocabulary, numeric imputation/scaling, and categorical encoding are all
learned inside `fit()`, which is exactly what you want after splitting data.

In [ ]:
classification_frame = reviews[
    [
        "review_text",
        "age",
        "positive_feedback_count",
        "department_name",
        "class_name",
        "recommended_ind",
    ]
].copy()

# The target is the real recommendation label from the review dataset.
# Stratification keeps the positive/negative balance similar in both splits.
train_df, test_df = train_test_split(
    classification_frame,
    test_size=0.25,
    random_state=42,
    stratify=classification_frame["recommended_ind"],
)

classifier = build_text_classifier(
    text_column="review_text",
    numeric_columns=["age", "positive_feedback_count"],
    categorical_columns=["department_name", "class_name"],
    max_features=2500,
    ngram_range=(1, 2),
    min_df=1,
    stop_words="english",
)

classifier.fit(train_df, train_df["recommended_ind"])
metrics = evaluate_classifier(classifier, test_df, test_df["recommended_ind"])

{key: value for key, value in metrics.items() if key not in {"report", "predictions", "confusion_matrix"}}

{'accuracy': 0.7428571428571429,
 'balanced_accuracy': 0.625,
 'precision_macro': 0.6134259259259259,
 'recall_macro': 0.625,
 'f1_macro': 0.6181818181818182,
 'f1_weighted': 0.7490909090909091}

In [ ]:
# The confusion matrix keeps the model honest. The rows are true labels and
# the columns are predicted labels.
pd.DataFrame(
    metrics["confusion_matrix"],
    index=["true_not_recommended", "true_recommended"],
    columns=["pred_not_recommended", "pred_recommended"],
)

,pred_not_recommended,pred_recommended
true_not_recommended,3,4
true_recommended,5,23


In [ ]:
# The report is returned as a dictionary so it can be logged, tested, or joined
# with experiment metadata.
pd.DataFrame(metrics["report"]).T.round(3)

,precision,recall,f1-score,support
0,0.375,0.429,0.400,7.000
1,0.852,0.821,0.836,28.000
accuracy,0.743,0.743,0.743,0.743
macro avg,0.613,0.625,0.618,35.000
weighted avg,0.756,0.743,0.749,35.000


In [ ]:
# Because the classifier is a normal scikit-learn pipeline, it can score new
# mixed-feature rows without manually repeating preprocessing steps.
new_reviews = pd.DataFrame(
    {
        "review_text": [
            "The fabric is soft, the cut is flattering, and I would recommend it.",
            "The sizing is poor and the zipper broke after one wear.",
        ],
        "age": [31, 45],
        "positive_feedback_count": [2, 8],
        "department_name": ["Dresses", "Dresses"],
        "class_name": ["Dresses", "Dresses"],
    }
)

pd.DataFrame(
    {
        "review_text": new_reviews["review_text"],
        "predicted_recommended": classifier.predict(new_reviews),
        "probability_recommended": classifier.predict_proba(new_reviews)[:, 1],
    }
)

,review_text,predicted_recommended,probability_recommended
0,"The fabric is soft, the cut is flattering, and I would recommend it.",0,0.431167
1,The sizing is poor and the zipper broke after one wear.,0,0.437322


## 10. Optional Transformer Classifier

`TransformerClassifier` is intentionally not run by default because it can
download model weights and requires PyTorch plus Hugging Face models. When
those are available, the same MAna interface gives you loaders, one-epoch
training, evaluation, prediction, and saving.

In [ ]:
RUN_TRANSFORMER_DEMO = 1

if RUN_TRANSFORMER_DEMO:
    from MAna.nlp import TransformerClassifier

    transformer = TransformerClassifier(
        model_name="distilbert-base-uncased",
        num_labels=2,
        max_length=160,
    )

    train_loader = transformer.make_loader(
        train_df["review_text"].tolist(),
        train_df["recommended_ind"].astype(int).tolist(),
        batch_size=8,
        shuffle=True,
    )

    test_loader = transformer.make_loader(
        test_df["review_text"].tolist(),
        test_df["recommended_ind"].astype(int).tolist(),
        batch_size=8,
    )

    first_epoch_loss = transformer.train_epoch(train_loader)
    transformer_metrics = transformer.evaluate(test_loader)
    transformer_predictions = transformer.predict(test_df["review_text"].head(5).tolist())

Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [ ]:
{
        "first_epoch_loss": first_epoch_loss,
        "balanced_accuracy": transformer_metrics["balanced_accuracy"],
        "sample_predictions": transformer_predictions["labels"].tolist(),
        "True_labels": test_df["recommended_ind"].head(5).tolist(),
    }

{'first_epoch_loss': 0.5705558336698092,
 'balanced_accuracy': 0.5,
 'sample_predictions': [1, 1, 1, 1, 1],
 'True_labels': [1, 1, 1, 1, 0]}

## 11. Practical Recipe

For a real project, the pieces usually connect like this:

1. Load with `read_data()` so source formats can change without rewriting the
   workflow.
2. Use `DataCleaner` for table-level hygiene and `TextCleaner` for text-level
   hygiene.
3. Use chunking when documents are long or when retrieval needs source-level
   traceability.
4. Use TF-IDF/count/hash vectorization for classical modeling and explainable
   baselines.
5. Use sentiment and topic modeling for fast corpus diagnostics.
6. Use embeddings and semantic search when users ask meaning-based questions.
7. Use `build_text_classifier()` when the target is known and mixed features
   matter.
8. Move to transformer classification when the project needs deeper language
   understanding and the infrastructure is ready.

That is the shape of `MAna.nlp`: start light, stay inspectable, and upgrade
only when the problem deserves it.